<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/hardness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Hardness removal***


In [ ]:
import math as mh
import pandas as pd
import numpy as np

def hardness(lanes,Partable2,dailyflow,Treatment):
    # Hardness removal  design
    # This code calculates the dimensions of hardness removal stage in a water treatment plant.
    #treatmenttechnologies2 = treatmenttechnologies.astype(str)

    # Input parameters
    print("")
    print("HARDNESS REMOVAL PROCESS")
    flowRate = dailyflow/24  # m3/h
    hardn = Partable2['Hardness'][2]
    print("The total hardness in the source water is",hardn)
    # Select method
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "pellet softening").any():
      softarea,softenergy,lime_dosage_kg_per_day,softheadloss,softheight =pellet(flowRate,lanes,hardn) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "cationic iex").any():
      softarea,softenergy,lime_dosage_kg_per_day,softheadloss,softheight=cationiciex(flowRate,lanes,hardn) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "nf/ro").any():
      softarea,softenergy=nfro(flowRate,lanes,hardn) #, units
      lime_dosage_kg_per_day,softheight= 0, 0
    else:
      softarea = 0
      softenergy = 0
      lime_dosage_kg_per_day=0
      softheight=0

    return softarea, softenergy,lime_dosage_kg_per_day,softheadloss,softheight

def pellet(flowRate,lanes,hardn):
    print("A minimum of two pellet reactors per lane is recommended.")
    units=float(input("Enter the number of units per lane: "))
    flowlane=flowRate/lanes # m3/h
    flowunit=flowRate/(lanes*units) # m3/h
    hardness_target_mg_L = float(input("set target hardness in mg/L as CaCO3:"))
    delta_hardness_mg_L = hardn - hardness_target_mg_L # hardness removal
    while True:
        try:
            if hardness_target_mg_L < hardn:
                print("The ΔC (mg/L)CaCO3 to be removed is ",delta_hardness_mg_L)
                break
            else:
                print("Target hardness is higher than hardness in the source water. Hardness in the source is",hardn)
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    pellet_porosity = float(input("Enter pellet bed porosity (0-1, default 0.4): ") or 0.4)
    pellet_density = float(input("Enter pellet density (kg/m3, default 2700): ") or 2700)

    try:

        min_height = float(input("Enter minimum height (m): "))
        max_height = float(input("Enter maximum height (m): "))
        min_diameter = min_height/2
        max_diameter = max_height/4
    except ValueError:
        print("Invalid input. Please enter numeric values.")
        exit()

    # Generate feasible dimensions within the provided ranges
    step=0.2
    diameters = np.arange(min_diameter, max_diameter + step, step)
    heights = np.arange(min_height, max_height + step, step)

    # Constants for calculation
    DENSITY_WATER = 1000  # kg/m^3 (density of water)
    RETENTION_TIME_MIN = 15  # minimum retention time in minutes
    RETENTION_TIME_MAX = 40  # maximum retention time in minutes

    # Constants for calculation
    results = []
    for diameter in diameters:
        for height in heights:
            # Calculate reactor volume (m^3)
            volume = np.pi * (diameter / 2) ** 2 * height

            # Calculate retention time based on volume and flow rate (minutes)
            retention_time_hydraulic = (volume / flowunit) * 60

            # Check if both retention times are within acceptable range
            if RETENTION_TIME_MIN <= retention_time_hydraulic <= RETENTION_TIME_MAX: #and retention_time_kinetic <= RETENTION_TIME_MAX:
                # Calculate hardness removal capacity (simplified assumption: proportional to flow rate)
                hardness_removed_unit = flowunit * delta_hardness_mg_L/1000  # mg/L to kg/h
                hardness_removed_lane=round(hardness_removed_unit*lanes*24/step)*step #kg/day
                hardness_removed = hardness_removed_lane * lanes #kg/day
                headlosses = -((1 - pellet_porosity) * height * (DENSITY_WATER - pellet_density) / DENSITY_WATER)

                # Store results
                results.append({
                    "Diameter (m)": round(diameter, 1),
                    "Height (m)": round(height, 1),
                    "Volume (m^3)": round(volume, 1),
                    "Hydraulic Retention Time (min)": round(retention_time_hydraulic, 1),
                    "Hardness Removed per lane (Kg/d)": round(hardness_removed_lane, 1),
                    "Hardness Removed (Kg/d)": round(hardness_removed, 1),
                    "Headlosses (m)": round(headlosses, 1)
                })

    if not results:
        print("No feasible reactor dimensions found within the given parameters.")
        return 0, 0, 0  # or some other default values

    # Sort results by hydraulic retention time for convenience
    results.sort(key=lambda x: x["Hydraulic Retention Time (min)"])

    # Display the options to the user
    print("Available solutions:")
    for idx, result in enumerate(results, start=1):
        print(f"\nOption {idx}:")
        for key, value in result.items():
            print(f"  {key}: {value}")

    # Prompt the user to select an option
    while True:
        try:
            user_choice = int(input("\nEnter the number of your chosen solution: ")) - 1
            if 0 <= user_choice < len(results):
                selected_solution = results[user_choice]
                print("\nYou selected:")
                for key, value in selected_solution.items():
                    print(f"  {key}: {value}")
                # Calculate the area based on the diameter
                diameter = selected_solution["Diameter (m)"]
                softheadloss=selected_solution["Headlosses (m)"]
                height=selected_solution["Height (m)"]
                softareareact = np.pi * (diameter / 2) ** 2
                print(f"\nThe area of the base for the selected reactor is: {softareareact:.2f} m²")
                break
            else:
                print("Invalid choice. Please enter a valid number.")
        except ValueError:
            print("Invalid input. Please enter a number.")

    softareareactall=round(softareareact*units*lanes)
    print(f"\nThe overall area of the softening reactors is: {softareareactall:.2f} m²")

    # Lime dosage
    lime_dosage_per_hardness_mg_L = 0.75  # Lime dosage per mg/L hardness removed (mg/L)
    print("Assumption: The lime dosage (mg/L) per mg/L hardness removed is 0.75")
    lime_dosage_mg_L = lime_dosage_per_hardness_mg_L * delta_hardness_mg_L
    lime_dosage_kg_per_day = (lime_dosage_mg_L * flowRate*24) / 1000
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    lime_dosage_kg_per_month = round(lime_dosage_kg_per_day * ndays)
    lime_density=2210 #kg/m3
    soft_storage_m3=lime_dosage_kg_per_month/lime_density
    softareachem=round(soft_storage_m3/2.2)
    print(f"\nThe area of the lime storage (m2) is,{softareachem:.2f} m²")
    # Sludge Pellet storage
    while True:
        ndays = int(input("Set the pellet storage in days (min 1day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    hardness_removed= hardness_removed*ndays
    sl_storage_m3=hardness_removed/pellet_density
    sl_storage_m2=round(soft_storage_m3/2.2)
    print(f"\nThe required area for the pellet storage is {sl_storage_m2:.2f} m²")

    #Total area
    softarea=round(softareachem+softareareactall+sl_storage_m2)
    print(f"\nThe required area for the softening unit is {softarea:.2f} m²")

    # Energy consumption
    lime_dosage_m3_per_h = round(lime_dosage_kg_per_day / (lime_density*24))
    Pdosagepumps=1.1*(lime_density*9.81*lime_dosage_m3_per_h/3600*(softheadloss+height))/(0.8*3600)
    softenergy = round(24*Pdosagepumps,1)
    print("The total energy consumption per day (kWh) is",softenergy)
    softheight=height

    return softarea,softenergy,lime_dosage_kg_per_day,softheadloss,softheight

def cationiciex(flowRate,lanes,hardn):

    print("A minimum of two Cationic IEX tanks per lane is required, one for the treatment process and one for regeneration of brine and backwash.")
    units=float(input("Enter the number of tanks per lane (including one in stand-by): "))
    while True:
        try:
            if units >= 2:
                print("The total number of service tanks in the DWTP is",units*lanes)
                break
            else:
                print("The minimum number of units per lane should be two")
        except ValueError:
            print("Invalid input. Please enter a valid number.")
    # Nominal flow
    flowlane=flowRate/lanes # m3/h
    flowunit=flowRate/(lanes*(units-1)) # m3/h

    # Hardness removal
    hardness_target_mg_L = float(input("set target hardness in mg/L as CaCO3:"))
    delta_hardness_mg_L = hardn - hardness_target_mg_L # hardness removal
    while True:
        try:
            if hardness_target_mg_L < hardn:
                print("The ΔC (mg/L)CaCO3 to be removed is ",delta_hardness_mg_L)
                break
            else:
                print("Target hardness is higher than hardness in the source water. Hardness in the source is",hardn)
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    print('1 equivalent weight CaCO₃ = 50 mg/meq')
    caequi=50
    hardness_removal_eq_l = delta_hardness_mg_L / caequi #meq/L
    hardness_removal_eq = hardness_removal_eq_l * flowunit*1000 #meq/h
    hardness_removal_eq = hardness_removal_eq/1000 #eq/h

    # Required resin capacity per unit
    raisin_cap=1.8 #eq/l
    service_run_hours=float(input("Enter the regeneration cycle (range 6 - 12 hours): "))
    total_exchange_cycle = hardness_removal_eq * service_run_hours
    V_resin_l_unit=total_exchange_cycle/raisin_cap
    V_resin_m3_unit=1.1* V_resin_l_unit/1000 #m3
    beddepth=float(input("Enter the bed height height(0.9m - 1.2m):"))
    softheight=beddepth+ 0.5*beddepth+0.3
    A_resin_m2_unit = V_resin_m3_unit/beddepth
    A_resin_area=1.2*round(A_resin_m2_unit*units*lanes,1)

    # Lime dosage
    mass_h=delta_hardness_mg_L * flowRate *1000 / 1000000  # kg/h CaCO₃
    caoh_caco3=0.74
    lime_dosage_kg_per_h = caoh_caco3 * mass_h
    lime_dosage_kg_per_day = round(lime_dosage_kg_per_h * 24,1)
    print("The lime dosage (kg/day) is",lime_dosage_kg_per_day)
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    lime_dosage_kg_per_month = round(lime_dosage_kg_per_day * ndays)
    lime_density=2210 #kg/m3
    soft_storage_m3=lime_dosage_kg_per_month/lime_density
    softareachem=round(soft_storage_m3/2.2)
    print(f"\nThe area of the lime storage (m2) is,{softareachem:.2f} m²")

    # Headlosses
    flow_m3_s = flowunit / 3600
    velocity_m_s = flow_m3_s / A_resin_m2_unit
    Re = (1000 * velocity_m_s * 0.0005) / 1e-3
    term1 = 150 * ((1 - 0.4)**2) / (Re * 0.4**3)
    term2 = 1.75 * ((1 - 0.4)) / (0.4**0.4)
    friction_factor = term1 + term2
    head_loss_m = friction_factor * (softheight / 0.0005) * (velocity_m_s**2 / (2 * 9.81))
    softheadloss = head_loss_m*1.2  # Add safety margin

    # Dosage pumps energy consumption
    lime_dosage_m3_per_h = round(lime_dosage_kg_per_day / (lime_density*24))
    print("The lime dosage (m3/h) is",lime_dosage_m3_per_h
    Pdosagepumps=1.1*(lime_density*9.81*lime_dosage_m3_per_h/3600*(softheadloss+softheight))/(0.8*3600)
    dosagesoftenergy = round(24*Pdosagepumps,1)
    print("The dossage pumps energy consumption per day (kWh) is",dosagesoftenergy)

    #Backwash calculations
    print("We assume one unit per lane backwashing per day and 15 minutes of backwashing time")
    Qbackunit=V_resin_m3_unit*5 #5 m/h velocity
    Vbackunit=Qbackunit*15/60
    Abackunit=Vbackunit/2.2 #m2
    Aback=round(Abackunit*units*lanes)
    print(f"\nThe required area for the backwash is {Aback:.2f} m²")
    Hback=1.1*(softheight+softheadloss)+20 #pressure of 20m for cleanning
    Pbackunit=2.725*Qbackunit*Hback/0.75
    Ebackcycle=Pbackunit*15/60
    Ebackday=Ebackcycle*3
    Ebackall=Ebackday*lanes
    print(Ebackall)
    #Total area required for the softening unit
    softarea=round(softareachem+A_resin_area+Aback)
    print(f"\nThe required area for the softening unit is {softarea:.2f} m²")

    #Total energy consumption
    softenergy=round(dosagesoftenergy+Ebackall)
    print("The total energy consumption per day (kWh) is",softenergy)

    return softarea,softenergy,lime_dosage_kg_per_day,softheadloss,softheight

def nfro(flowRate,lanes,hardn):
    """
    Calculates NF/RO system design parameters.

    Args:
        flow_Rate: Flow rate in m3/day
        flowlane: Flow rate per lane in m3/day
        flowunit: Flow rate per unit in m3/day
        delta_hardness_mg_L:  Hardness removal in mg/L
        initial_hardness: Initial hardness in mg/L
        nf_rejection: NF membrane rejection
        ro_rejection: RO membrane rejection
        nf_permeate_per_membrane: NF permeate flow area per element in m3/day
        ro_permeate_per_membrane: RO permeate flow area per element in m3/day
        operating_pressure_nf: Operating pressure for NF stage in bar
        operating_pressure_ro: Operating pressure for RO stage in bar
        pump_efficiency: Pump efficiency (decimal)
        membrane_orientation: 'vertical' or 'horizontal'
        membranes_per_vessel: Membranes per vessel (set 6 - 8 per vessel)

    Returns:
        A dictionary containing:
            - nf_membranes: Number of NF membranes
            - ro_membranes: Number of RO membranes
            - nf_membrane_area: Total NF membrane area
            - ro_membrane_area: Total RO membrane area
            - nf_permeate_flow: NF permeate flow rate
            - ro_permeate_flow: RO permeate flow rate
            - energy_consumption_nf: Energy consumption for NF stage (kWh/day)
            - energy_consumption_ro: Energy consumption for RO stage (kWh/day)
            - concentration_polarization_factor: Estimated concentration polarization factor (simplified)
            - scaling_potential: Simplified scaling potential indicator (higher values indicate higher risk)
    """

    units=float(input("Enter the number of units per lane(minimum 2 units): "))
    while True:
        try:
            if units >= 1:
                print("The total number of NF/RO units in the DWTP is",units*lanes)
                break
            else:
                print("One unit per lane is required")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Nominal flow
    flowlane=flowRate/lanes # m3/h
    flowunit=flowRate/(lanes*(units-1)) # m3/h
    flow_rate=flowunit*24 #m3/day
    # Hardness removal
    hardness_target_mg_L = float(input("set target hardness in mg/L as CaCO3:"))
    delta_hardness_mg_L = hardn - hardness_target_mg_L # hardness removal
    while True:
        try:
            if hardness_target_mg_L < hardn:
                print("The ΔC (mg/L)CaCO3 to be removed is ",delta_hardness_mg_L)
                break
            else:
                print("Target hardness is higher than hardness in the source water. Hardness in the source is",hardn)
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Calculate permeate flow rates
    nf_recovery=float(input("Set the NF rejection rate (estimated between 0.7-0.9):"))
    ro_recovery=float(input("Set the RO rejection rate (estimated between 0.9-0.95):"))
    nf_permeate_flow = flow_rate * nf_recovery
    ro_permeate_flow = flow_rate * ro_recovery
    print("The NF permeate flow rate (m3/day) is",nf_permeate_flow)
    print("The RO permeate flow rate (m3/day) is",ro_permeate_flow)
    con_flow=flow_rate-ro_permeate_flow

    # Calculate number of membranes
    print("The dimensions of the membranes is set using commercial membranes specifications of two main commercial brands Nitto and CSM")
    print("The selected nanofiltration (NF) membrane has a length A of 1.1 meters, a width of 0.2 meters and feed pipe of 30 mm")
    Jc_nf=int(input("The Flux Rate of the NF system is (15-30 L/m2*h):"))
    print("The selected reverse osmosis (RO) membrane has a length A of 1.1 meters, a width of 0.2 meters and feed pipe of 20 mm")
    Jc_ro=int(input("The Flux Rate of the RO system is (10-20 L/m2*h):"))
    print("The active area per membrane is 37.2m2 for NF membranes and 40.9 per RO membranes")
    nf_area_per_membrane=37.2
    ro_area_per_membrane=40.9
    nf_area_oveall=nf_permeate_flow*41.7/Jc_nf
    ro_area_oveall=ro_permeate_flow*41.7/Jc_ro
    nf_membranes = mh.ceil(nf_area_oveall / nf_area_per_membrane)
    ro_membranes = mh.ceil(ro_area_oveall / ro_area_per_membrane)

    # Calculate energy consumption (kWh/day)
    # 1 bar = 100 kPa
    print("The operating pressure of the NF membranes is 6 bar")
    print("The operating pressure of the RO membranes is 12 bar")
    operating_pressure_nf=6
    operating_pressure_ro=12
    pump_efficiency=0.75
    nf_power = (nf_permeate_flow *9.81) * (operating_pressure_nf * 10.2) / (86400 * pump_efficiency)  # kW
    ro_power = (ro_permeate_flow *9.81) * (operating_pressure_ro * 10.2) / (86400 * pump_efficiency)  # kW
    energy_consumption_nf = round(nf_power * 24)  # kWh/day
    energy_consumption_ro = round(ro_power * 24)  # kWh/day
    print("The energy consumption (KWh) for the NF stage is",energy_consumption_nf)
    print("The energy consumption (KWh) for the RO stage is",energy_consumption_ro)
    softenergyunit=round(energy_consumption_nf+energy_consumption_ro)
    print("The total energy consumption per unit per day (kWh) is",softenergyunit)
    softenergy=round(softenergyunit*units*lanes)
    print("The total energy consumption per day (kWh) is",softenergy)

    # Estimate concentration polarization factor (simplified) - This requires more detailed modeling
    membrane_orientation=input("Enter the membrane orientation (vertical or horizontal):")
    if membrane_orientation == 'vertical':
        concentration_polarization_factor = 1.2  # Example value, adjust based on actual flow conditions
    elif membrane_orientation == 'horizontal':
        concentration_polarization_factor = 1.5  # Example value, adjust based on actual flow conditions
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")

    # Simplified scaling potential indicator (higher values indicate higher risk)
    scaling_potential = (hardn - hardness_target_mg_L) * concentration_polarization_factor

    #Set up the membranes per vessel
    membranes_per_vessel=int(input("Set the number of membranes per vessel (6-8):"))
    nf_no_vesels=mh.ceil(nf_membranes/membranes_per_vessel)
    ro_no_vesels=mh.ceil(ro_membranes/membranes_per_vessel)

    # Membrane characteristics
    print("NF Membranes:", nf_membranes)
    print("RO Membranes:", ro_membranes)
    print("NF Membrane Area:", nf_area_per_membrane, "m2")
    print("RO Membrane Area:", ro_area_per_membrane, "m2")
    print("NF Permeate Flow:", nf_permeate_flow, "m3/day")
    print("RO Permeate Flow:", ro_permeate_flow, "m3/day")
    print("NF Energy Consumption:", energy_consumption_nf, "kWh/day")
    print("RO Energy Consumption:", energy_consumption_ro, "kWh/day")
    print("Concentration Polarization Factor:", concentration_polarization_factor)
    print("Scaling Potential:", scaling_potential)
    print("Number of vessels in the NF process:", nf_no_vesels)
    print("Number of vessels in the RO process:", ro_no_vesels)

    # Soft area volume
    if membrane_orientation == 'vertical':
       nf_no_vesels_area=1.1*(0.3*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))*nf_no_vesels
       ro_no_vesels_area=1.1*(0.3*((membranes_per_vessel*0.25)+((membranes_per_vessel-1)*0.2)))*ro_no_vesels
       softareaunit=nf_no_vesels_area+ro_no_vesels_area
       pump_unit=round(1.2*ompunit)
       print("pump area:",pump_unit)
       softarea=round((softareaunit+pump_unit)*units*lanes)
       print("The total area of the NF/RO process (m2) is:",softarea)
    elif membrane_orientation == 'horizontal':
       nf_no_vesels_area=1.1*(1.2*((nf_no_vesels*0.25)+((nf_membranes-1)*0.2)))
       ro_no_vesels_area=1.1*(1.2*((ro_no_vesels*0.25)+((ro_membranes-1)*0.2)))
       softareaunit=nf_no_vesels_area+ro_no_vesels_area
       pump_unit=round(1.2*ompunit)
       print("pump area:",pump_unit)
       softarea=round((softareaunit+pump_unit)*units*lanes)
       print("The total area of the NF/RO process (m2) is:",softarea)
    else:
        raise ValueError("Invalid membrane orientation. Choose 'vertical' or 'horizontal'.")

    return softarea,softenergy